In [ ]:
from valdpy import ValdAuth, DynamoAPI
from valdpy.utils import read_credentials

%load_ext autoreload
%autoreload 2

# Dynamo API Example

This example demonstrates how to use the VALDPY package to access Dynamo (jump/power) test data.

In [ ]:
# Read credentials from file
creds = read_credentials('vald_api_cred.txt')

client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']

print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

## Step 1: Authentication

In [ ]:
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

In [ ]:
token = auth.get_token()
print(f"Token obtained: {token[:20]}...")

### Optional: Get Tenant Information

In [ ]:
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")

In [ ]:
# Get tenant info
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

## Step 2: Get Categories and Groups

In [ ]:
categories_df = auth.get_tenant_categories()
print(categories_df[['name', 'id']].to_string(index=False))

In [ ]:
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].head(10).to_string(index=False))

## Step 3: Get Profiles

In [ ]:
group_name = 'Research'
category_name = 'Team'

try:
    profiles_df = auth.get_group_profiles(group_name=group_name, category_name=category_name)
    print(f"Found {len(profiles_df)} profile(s)")
    print(profiles_df[['firstName', 'lastName', 'profileId']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error: {e}")

## Step 4: Initialize Dynamo API

In [ ]:
dynamo = DynamoAPI(tenant_id=auth.tenant_id, header=auth.header, region='USA')

### Get Tests Between Dates

Query Dynamo tests within a date range (max 180 days).

In [ ]:
start_date = '01/01/2025'
end_date = '31/01/2025'

tests_df = dynamo.get_tests(start_date, end_date)

if tests_df is not None:
    print(f"Found {len(tests_df)} test(s)")
    print(tests_df[['testId', 'profileId']].head())
else:
    print("No tests found in date range")

In [ ]:
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    results_df = dynamo.get_test_results(test_id)
    
    if results_df is not None:
        print(f"Test Results Shape: {results_df.shape}")
        print(results_df.head())
    else:
        print("No results found")
else:
    print("No tests available")

### Step 2 - Get individual test results
Using "get_test_results()", a test ID is used to pull the results for each rep within a denoted test. The resulting dataframe contains a row for each result from each rep.
Some what obsolete a get_test() returns test results as well.

In [ ]:
testID = dynamo.tests_df.loc[0,'id']
dynamo.get_test_results(testID)
dynamo.tests_df.head()

### Step 3 - Get the force trace for a given test
Using "get_force_trace()" will return the force trace in a dataframe for a given test ID.

In [ ]:
dynamo.get_force_trace(testID)
dynamo.raw['forceTrace'].head()

In [ ]:
# To access raw force trace within API call response
dynamo.raw['forceTrace'].plot(x='timeSeconds',y='forceNewtons',title='DynamoForce Trace',xlabel='Time (s)',ylabel='Force (N)')